#### DuckDB : Requêtage direct de fichiers Zero-Copy
- Interroger directement des fichiers (CSV, Parquet, JSON, etc.) sans les charger au préalable
- Exécuter des opérations SQL directement sur le fichier Parquet

In [1]:
import duckdb as d

In [2]:
parquet_filepath = '../00_data_sources/yellow_tripdata_2023-01.parquet'

taxi_data_202301 = d.read_parquet(parquet_filepath)

print(taxi_data_202301.limit(5))


┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ airport_fee │
│  int64   │      timestamp       │       timestamp       │     double      │    double     │   double   │      varchar       │    int64     │    int64     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │
├──────────┼──────────────────────┼───────────────────────┼───────────

In [3]:
d.sql("DESCRIBE FROM taxi_data_202301").show()

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ VendorID              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_distance         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ RatecodeID            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_and_fwd_flag    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID          │ BIGINT      │ 

In [4]:
result = d.sql(
    """
    SELECT COUNT(*) AS total_trips FROM taxi_data_202301
    """
)
print(result)

┌─────────────┐
│ total_trips │
│    int64    │
├─────────────┤
│     3066766 │
└─────────────┘



In [6]:
result2 = d.sql(
    """
    SELECT COUNT(*) AS total_trips,
    AVG(trip_distance) AS avg_distance,
    SUM(total_amount) AS total_revenue
    FROM taxi_data_202301
    WHERE total_amount > 0 AND trip_distance > 0;
    """
)
print(result2)

┌─────────────┬───────────────────┬──────────────────┐
│ total_trips │   avg_distance    │  total_revenue   │
│    int64    │      double       │      double      │
├─────────────┼───────────────────┼──────────────────┤
│     2998642 │ 3.910143858453183 │ 82033717.9997885 │
└─────────────┴───────────────────┴──────────────────┘

